# Huấn luyện YOLOv5 n/s · lượt so sánh chính · Signa v1.1

YOLOv5 **bản gốc dùng anchor** từ [ultralytics/yolov5](https://github.com/ultralytics/yolov5/tree/1e7cf74cb64d719b34f7315aa13eab9e32cde672) ghim commit.
Chạy ở **runtime riêng** với notebook YOLOv8, vì repo này cần `ultralytics>=8.4` còn YOLOv8 ghim 8.3.203.

Cùng giao thức với YOLOv8: ảnh 640, khởi tạo COCO, tối đa 200 epoch, patience 30, seed 0,
[bảng tăng cường chung](https://github.com/hana200804/DA/blob/f4ad6edc24d26f0cb6d937c8331ba48f98216234/ai/configs/augment.md) qua `configs/yolov5-hyp.yaml` và `patch_yolov5()`.
Chọn `best.pt` và dừng sớm theo mAP@0.5:0.95 (YOLOv5 mặc định pha 10% mAP50, notebook đổi lại cho giống YOLOv8).

1. Runtime → Change runtime type → **L4 GPU** (hoặc T4).
2. `signa-v1.1.zip` nằm ở thư mục gốc My Drive.
3. Lấy SHA từ liên kết cố định GitHub của bản notebook đang chạy (`…/blob/<SHA>/…`). Thêm một ô mã ở đầu notebook trên Colab với `%env SIGNA_NOTEBOOK_COMMIT=<SHA>` và thay `<SHA>` bằng giá trị đó.
4. **Runtime → Run all.** Train `yolov5n` rồi `yolov5s`, xuất dự đoán val, chấm bằng bộ đánh giá chung.
5. Mất phiên: **Run all** lần nữa. Run đã xong được bỏ qua, run dở tiếp tục từ `last.pt`.

Kết quả ở `My Drive/Signa/experiments/yolov5{n,s}-640-v1.1-main-s0/`. Không xuất test.

## 1. Cài đặt

In [ ]:
import json
import os
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("SIGNA_SMOKE") == "1"  # 1 epoch, 2% train, ghi ra /content
REPO_COMMIT = "f4ad6edc24d26f0cb6d937c8331ba48f98216234"
# Đặt SIGNA_NOTEBOOK_COMMIT theo SHA trên permalink GitHub của notebook đang chạy.
NOTEBOOK_COMMIT = os.environ.get("SIGNA_NOTEBOOK_COMMIT") or None
REPO = Path(os.environ.get("SIGNA_REPO", "/content/signa-source"))


def sh(*args):
    subprocess.run([str(a) for a in args], check=True)


if IN_COLAB:
    packages = ["albumentations==2.0.8", "pycocotools"]
    sh(sys.executable, "-m", "pip", "install", "-q", *packages)
    if not (REPO / ".git").exists():
        sh("git", "clone", "-q", "https://github.com/hana200804/DA.git", REPO)
    sh("git", "-C", REPO, "fetch", "-q", "origin")
    sh("git", "-C", REPO, "checkout", "-q", "--detach", REPO_COMMIT)
sys.path.insert(0, str(REPO / "ai/src"))

YOLOV5_COMMIT = "1e7cf74cb64d719b34f7315aa13eab9e32cde672"
YOLOV5 = Path(os.environ.get("SIGNA_YOLOV5", "/content/yolov5"))
if IN_COLAB:
    if not (YOLOV5 / ".git").exists():
        sh("git", "clone", "-q", "https://github.com/ultralytics/yolov5.git", YOLOV5)
    sh("git", "-C", YOLOV5, "checkout", "-q", "--detach", YOLOV5_COMMIT)
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", YOLOV5 / "requirements.txt")
# Repo gốc dùng anchor (docs/04 §2), không dùng yolov5u của gói ultralytics.
sys.path.insert(0, str(YOLOV5))

## 2. Dữ liệu

In [ ]:
import hashlib
from zipfile import ZipFile

import torch
import yaml

from signa_ai.dataset import read_yolo, relative_path
from signa_ai.detection import EVAL_CONFIDENCE, Box, Detection, to_coco_predictions
from signa_ai.labels import load_label_map

assert torch.cuda.is_available(), "Chọn GPU (L4 hoặc T4) trong Runtime → Change runtime type."
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU)

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    ARCHIVE = Path("/content/drive/MyDrive/signa-v1.1.zip")
    DATASET = Path("/content/v1.1")
    # Smoke ghi ra /content để không lẫn với run thật trên Drive.
    DRIVE_ROOT = Path("/content/drive/MyDrive/Signa/experiments")
    if SMOKE:
        DRIVE_ROOT = Path("/content/signa-smoke")
    assert ARCHIVE.is_file(), f"Thiếu {ARCHIVE}"
    if not DATASET.exists():
        with ZipFile(ARCHIVE) as archive:
            for member in archive.namelist():
                root = relative_path(member).parts[0]
                assert root == "v1.1", f"ZIP sai thư mục gốc: {member}"
            archive.extractall("/content")
else:
    DATASET = Path(os.environ["SIGNA_DATASET"])
    DRIVE_ROOT = Path(os.environ["SIGNA_RUNS"])

label_map = load_label_map(REPO / "ai/data/labels.yaml")
assert label_map.version == 1 and len(label_map) == 46
NAMES = label_map.names
config_path = Path("/content/signa-data.yaml") if IN_COLAB else DRIVE_ROOT / "data.yaml"
config = yaml.safe_load((DATASET / "data.yaml").read_text())
assert config["names"] == NAMES, "data.yaml lệch danh mục v1."
config["path"] = str(DATASET)
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
assert (DATASET / "manifest.csv").read_bytes() == (REPO / "ai/data/manifests/v1.1.csv").read_bytes()

EXPECTED = {"train": (6679, 13283), "val": (1599, 2685), "test": (1700, 2664)}
GROUND_TRUTH = {}
for split, (image_count, box_count) in EXPECTED.items():
    assert config[split] == f"images/{split}"
    truth = json.loads((DATASET / f"coco/{split}.json").read_text())
    assert truth["info"]["label_map_version"] == label_map.version
    assert {c["id"]: c["name"] for c in truth["categories"]} == NAMES
    assert len(truth["images"]) == image_count and len(truth["annotations"]) == box_count
    boxes = 0
    for image in truth["images"]:
        rel = relative_path(image["file_name"])
        assert rel.parts[:2] == ("images", split) and (DATASET / rel).is_file(), str(rel)
        label = DATASET / "labels" / split / rel.with_suffix(".txt").name
        boxes += len(read_yolo(label, label_map))
    assert boxes == box_count
    GROUND_TRUTH[split] = truth
    print(split, image_count, "ảnh,", boxes, "box")

## 3. Cấu hình

In [ ]:
import csv

import numpy as np
import train as yolov5_train
from models.common import AutoShape, DetectMultiBackend
from utils.callbacks import Callbacks
from utils.torch_utils import select_device

from signa_ai.augment import patch_yolov5

os.environ['PYTHONPATH'] = str(REPO / 'ai/src')  # cho subprocess signa_ai.evaluate
patch_yolov5()  # pipeline Albumentations chung thay pipeline mặc định


def map50_95_fitness(x):
    """Chọn best.pt và dừng sớm theo mAP@0.5:0.95 như YOLOv8, DETR.

    Mặc định YOLOv5 pha 10% mAP50.
    """
    return x[:, 3]


yolov5_train.fitness = map50_95_fitness
assert yolov5_train.fitness(np.array([[1, 1, 0.9, 0.5, 0, 0, 0]]))[0] == 0.5

SIZES = ("n", "s")
SEED = 0
EPOCHS = 1 if SMOKE else 200
BATCH = 16
HYP = REPO / "ai/configs/yolov5-hyp.yaml"
data_path = config_path
if SMOKE:  # 2% ảnh train để kiểm thử nhanh
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    subset = sorted((DATASET / "images/train").iterdir())[::50]
    (DRIVE_ROOT / "train-smoke.txt").write_text("\n".join(str(p) for p in subset))
    smoke = dict(config, train=str(DRIVE_ROOT / "train-smoke.txt"))
    data_path = DRIVE_ROOT / "data-smoke.yaml"
    data_path.write_text(yaml.safe_dump(smoke, allow_unicode=True, sort_keys=False))


def run_name(size):
    return f"yolov5{size}-640-v1.1-main-s{SEED}"


def train_args(size):
    return dict(
        weights=f"yolov5{size}.pt",
        data=str(data_path),
        hyp=str(HYP),
        epochs=EPOCHS,
        batch_size=BATCH,
        imgsz=640,
        device="0",
        workers=2,
        patience=30,
        seed=SEED,
        project=str(DRIVE_ROOT),
        name=run_name(size),
        exist_ok=True,
    )


def write_best_selection(run_dir):
    with (run_dir / "results.csv").open() as stream:
        rows = [{k.strip(): v.strip() for k, v in row.items()} for row in csv.DictReader(stream)]
    best = max(rows, key=lambda r: float(r["metrics/mAP50-95"]))
    selected = {
        "epoch": int(best["epoch"]) + 1,
        "fitness": float(best["metrics/mAP50-95"]),
        "val": {k: float(v) for k, v in best.items() if k.startswith("metrics/")},
        "epochs_run": len(rows),
    }
    (run_dir / "best-selection.json").write_text(json.dumps(selected, indent=2))
    return selected


def predict_with(best_path):
    model = AutoShape(DetectMultiBackend(str(best_path), device=select_device("0")))
    model.conf, model.iou, model.max_det = EVAL_CONFIDENCE, 0.7, 300
    assert model.names == NAMES

    def predict_split(truth):
        predictions = []
        for image in truth["images"]:
            result = model(str(DATASET / image["file_name"]), size=640)
            assert result.ims[0].shape[:2] == (image["height"], image["width"])
            detections = [
                Detection(Box.from_xyxy(x1, y1, x2, y2), NAMES[int(cls)], float(score))
                for x1, y1, x2, y2, score, cls in result.xyxy[0].cpu().tolist()
            ]
            predictions.extend(to_coco_predictions(image["id"], detections, label_map))
        return predictions

    return predict_split

In [ ]:
from importlib.metadata import distributions


def freeze(run_dir):
    packages = {(d.metadata["Name"], d.version) for d in distributions()}
    lines = sorted(f"{name}=={version}" for name, version in packages if name)
    (run_dir / "requirements.txt").write_text("\n".join(lines) + "\n")


def run_status(run_dir):
    """done: đã đánh giá; trained: train xong, chưa đánh giá; resume: còn optimizer; new."""
    if (run_dir / "done.json").exists():
        return "done"
    last = run_dir / "weights/last.pt"
    if not last.is_file():
        return "new"
    # Chỉ nạp checkpoint do nhóm tạo; checkpoint PyTorch có thể thực thi mã khi nạp.
    ckpt = torch.load(last, map_location="cpu", weights_only=False)
    return "resume" if ckpt.get("optimizer") is not None else "trained"


def start_session(run_dir, extra):
    meta_path = run_dir / "meta.json"
    notebook = {"notebook_commit": NOTEBOOK_COMMIT}
    if NOTEBOOK_COMMIT is None:
        notebook["notebook_commit_note"] = "Chưa cung cấp SHA qua SIGNA_NOTEBOOK_COMMIT."
    meta = (
        json.loads(meta_path.read_text())
        if meta_path.exists()
        else {
            "repo_commit": REPO_COMMIT,
            **notebook,
            "dataset_version": "v1.1",
            "label_map_version": label_map.version,
            "augmentation": "signa_ai.augment (configs/augment.md)",
            "sessions": [],
        }
    )
    session = {
        "started_at": datetime.now(UTC).isoformat(),
        "gpu": GPU,
        "torch": torch.__version__,
        **notebook,
        **extra,
    }
    meta["sessions"].append(session)
    start = time.monotonic()
    save_session(run_dir, meta, session, start)
    return meta, session, start


def save_session(run_dir, meta, session, start, **fields):
    session["gpu_hours"] = (time.monotonic() - start) / 3600
    session.update(fields)
    # Giữ bản meta gần nhất nếu phiên bị ngắt giữa lúc ghi.
    pending = run_dir / "meta.json.tmp"
    pending.write_text(json.dumps(meta, ensure_ascii=False, indent=2))
    pending.replace(run_dir / "meta.json")


def train_with_tracking(run_dir, **kwargs):
    # run() dùng cùng tùy chọn nhưng không nhận callback.
    opt = yolov5_train.parse_opt(True)
    for key, value in kwargs.items():
        setattr(opt, key, value)
    meta, session, start = start_session(run_dir, {"yolov5_commit": YOLOV5_COMMIT})

    def on_model_save(last, epoch, final_epoch, best_fitness, fi):
        save_session(run_dir, meta, session, start, epoch=epoch + 1)

    callbacks = Callbacks()
    callbacks.register_action("on_model_save", callback=on_model_save)
    try:
        yolov5_train.main(opt, callbacks)
    finally:
        save_session(run_dir, meta, session, start)


def write_predictions_and_evaluate(run_dir, best_path, predict_split, export_test=False):
    """Xuất COCO JSON (conf 0,001) và chấm bằng bộ đánh giá chung. Test chỉ khi đã chốt."""
    evaluation = run_dir / "evaluation"
    evaluation.mkdir(exist_ok=True)
    for split in ("val", "test") if export_test else ("val",):
        predictions = predict_split(GROUND_TRUTH[split])
        output = evaluation / f"predictions-{split}.json"
        output.write_text(json.dumps(predictions, ensure_ascii=False, allow_nan=False))
        print(split, len(predictions), "dự đoán →", output)
    (evaluation / "export-config.json").write_text(
        json.dumps(
            {
                "datasetVersion": "v1.1",
                "labelMapVersion": label_map.version,
                "weights_sha256": hashlib.sha256(best_path.read_bytes()).hexdigest(),
                "conf": EVAL_CONFIDENCE,
                "iou": 0.7,
                "max_det": 300,
                "imgsz": 640,
                "test_exported": export_test,
            },
            indent=2,
        )
    )
    sh(
        sys.executable, "-m", "signa_ai.evaluate",
        "--gt", DATASET / "coco/val.json",
        "--pred", evaluation / "predictions-val.json",
        "--manifest", DATASET / "manifest.csv",
        "--labels", REPO / "ai/data/labels.yaml",
        "--out", run_dir / "metrics.json",
    )
    metrics = json.loads((run_dir / "metrics.json").read_text())
    summary = {k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}
    print(json.dumps(summary, indent=2))

## 4. Huấn luyện, xuất dự đoán val, chấm điểm

In [ ]:
for size in SIZES:
    run_dir = DRIVE_ROOT / run_name(size)
    status = run_status(run_dir)
    print(f"== yolov5{size}: {status}")
    if status in ("new", "resume"):
        run_dir.mkdir(parents=True, exist_ok=True)
        if status == "new":
            args = train_args(size)
            run_config = {
                "dataset_version": "v1.1",
                "label_map_version": label_map.version,
                **args,
            }
            (run_dir / "config.yaml").write_text(yaml.safe_dump(run_config, sort_keys=False))
            freeze(run_dir)
        else:
            args = {"resume": str(run_dir / "weights/last.pt")}
        train_with_tracking(run_dir, **args)
    if status != "done":
        selected = write_best_selection(run_dir)
        print("epoch chọn:", selected["epoch"], "mAP50-95 val:", selected["fitness"])
        best_path = run_dir / "weights/best.pt"
        write_predictions_and_evaluate(run_dir, best_path, predict_with(best_path))
        finished = {"finished_at": datetime.now(UTC).isoformat()}
        (run_dir / "done.json").write_text(json.dumps(finished))
        torch.cuda.empty_cache()
print("Xong. Kết quả trên Drive:", DRIVE_ROOT)